# 2 · Quem são os beneficiários dos rendimentos da dívida?

**Pergunta:** quando a União paga juros, para quem vai esse rendimento?

**Do que há evidência pública:**
- composição do estoque da DPMFi **por grupo de detentor** (Anexo 2.7 do RMD, série desde 2007);
- estimativa do **fluxo de juros por grupo** (juros nominais do ano × participação no estoque).

**O que NÃO é público:** nome por nome / instituição por instituição. O Requerimento
696/2019 (Câmara dos Deputados), que pedia essa lista ao STN, foi respondido com
**negativa por sigilo bancário**. Qualquer "banqueiro X recebeu Y bilhões" publicado
em rede social, sem fonte primária, é extrapolação.


In [1]:
import sys
from pathlib import Path
p = Path.cwd()
while not (p / "src").exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

from src.viz.graficos import (
    estilo, salvar, AZUL, VERMELHO, VERDE, LARANJA, ROXO, CINZA, CORES_DETENTOR,
)
from src.processa import indicadores as ind

estilo()
print("diretório de dados:", p / "dados" / "processado")


diretório de dados: /home/vanessa/devel/juros_e_divida/dados/processado


In [2]:
dz = ind.detentores_anuais()
part = dz.pivot_table(index="data", columns="detentor", values="participacao") * 100
part.tail(3).round(1)

detentor,Fundos de Investimento,Governo,Instituições Financeiras,Não-residentes,Outros,Previdência,Seguradoras,Tesouro Direto
data,,,,,,,,
2023-12-31,23.50,3.80,29.70,9.50,4.50,23.00,4.00,2.00
2024-12-31,21.70,3.40,29.50,10.20,5.10,23.90,4.00,2.30
2025-12-31,20.80,2.90,32.90,10.30,4.30,22.80,3.50,2.60


In [3]:
ordem = ["Instituições Financeiras", "Fundos de Investimento", "Previdência",
         "Não-residentes", "Tesouro Direto", "Seguradoras", "Governo", "Outros"]
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.stackplot(part.index, [part[c].to_numpy() for c in ordem],
             labels=ordem, colors=[CORES_DETENTOR[c] for c in ordem], alpha=0.85)
ax.set_ylabel("% do estoque DPMFi")
ax.set_ylim(0, 100)
ax.set_title("Quem detém a dívida mobiliária federal em mercado (2007–2026)")
ax.legend(loc="upper center", ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.12))
salvar(fig, "02_detentores_area")

fig -> viz/figs/02_detentores_area.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/02_detentores_area.png')

In [4]:
dez = dz[dz.data == dz.data.max()].sort_values("valor_bi", ascending=True)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(dez.detentor, dez.valor_bi, color=[CORES_DETENTOR[c] for c in dez.detentor])
for i, (v, pr) in enumerate(zip(dez.valor_bi, dez.participacao)):
    ax.text(v + 30, i, f"R$ {v:,.0f} bi ({pr:.1%})", va="center", fontsize=9)
ax.set_xlabel("R$ bilhões (estoque em dezembro do último ano completo)")
ax.set_xlim(0, dez.valor_bi.max() * 1.32)
ax.set_title("Estoque da DPMFi por detentor (dezembro)")
salvar(fig, "02_detentores_estoque")

fig -> viz/figs/02_detentores_estoque.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/02_detentores_estoque.png')

In [5]:
jf = ind.juros_por_detentor()
jf = jf[jf.ano < 2026]
conv = jf.pivot_table(index="ano", columns="detentor", values="juros_estimado_bi")
print("Juros nominais federais por ano (R$ bi):")
v = ind.juros_anuais()
ref = v[v.ano < 2026].set_index("ano")["juros_mi"] / 1000
print(ref.tail(12).round(0).to_string())

Juros nominais federais por ano (R$ bi):
ano
2014     311.00
2015     502.00
2016     407.00
2017     401.00
2018     379.00
2019     367.00
2020     312.00
2021     448.00
2022     586.00
2023     718.00
2024     950.00
2025   1,008.00


In [6]:
conv = conv.dropna(axis=1, how="all")
fig, ax = plt.subplots(figsize=(11, 5.5))
b = conv.loc[2015:]
ax.stackplot(b.index, [b[c].to_numpy() for c in b.columns],
             labels=list(b.columns),
             colors=[CORES_DETENTOR.get(c, "#999999") for c in b.columns], alpha=0.85)
ax.set_ylabel("R$ bilhões/ano (estimado)")
ax.set_title("Fluxo de juros estimado por grupo detentor (participação × juros nominais)")
ax.legend(loc="upper center", ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.12))
salvar(fig, "02_juros_por_detentor")

fig -> viz/figs/02_juros_por_detentor.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/02_juros_por_detentor.png')

**Premissa do gráfico acima:** participação do estoque em **dezembro** de cada
ano × juros nominais do ano (SGS 4616). Não é dado oficial de fluxo por detentor —
a União não publica quem recebeu; é a **melhor aproximação possível com dados
públicos**, boa para o tamanho relativo dos beneficiários.

## Quem segura o quê: o risco por título (RMD 2.8)

O anexo 2.7 responde **quem** recebe; o 2.8 responde **qual risco cada grupo
carrega** — a mesma dívida tem títulos de remunerações diferentes (LFT =
Selic; NTN-B = IPCA + juro real; LTN/NTN-F = pré-fixados) e quem segura cada
um não é o mesmo. Anexo 2.8 do RMD, desde 2011.


In [7]:
dt = pd.read_parquet(p / "dados" / "processado" / "rmd_detentores_titulo.parquet")
rotulos = {"Inst. Financeiras": "Instituições Financeiras",
           "Fundos": "Fundos de Investimento",
           "Não-residente": "Não-residentes"}
dt["detentor"] = dt.detentor.replace(rotulos)
ult = dt[dt.data == dt.data.max()]
compos = ult.pivot_table(index="detentor", columns="titulo", values="participacao")
compos = compos.reindex(ordem)[["LFT", "LTN", "NTN-B", "NTN-F"]]
print(f"Participação por título, {ult.data.max():%m/%Y} (%):")
print((compos * 100).round(1).to_string())


Participação por título, 09/2026 (%):
titulo                     LFT   LTN  NTN-B  NTN-F
detentor                                          
Instituições Financeiras 37.40 43.60  11.60  30.20
Fundos de Investimento   32.30 10.10  15.50   6.00
Previdência              17.40  7.60  47.00   6.90
Não-residentes            4.20 26.40   1.90  40.20
Tesouro Direto            2.30  2.60   6.10   0.90
Seguradoras               1.90  1.70   6.10   3.70
Governo                   0.30  5.20   4.30   8.20
Outros                    4.20  2.80   7.50   3.80


In [8]:
fig, ax = plt.subplots(figsize=(10, 5))
base = compos.drop(index="Outros", errors="ignore")
bottom = None
for det in [c for c in ordem if c in base.index]:
    vals = base.loc[det] * 100
    ax.bar(base.columns, vals, bottom=bottom, label=det,
           color=CORES_DETENTOR.get(det, "#999999"), alpha=0.9)
    bottom = vals if bottom is None else bottom + vals
ax.set_ylabel("% do estoque do título")
ax.set_title("Quem segura cada título da DPMFi (RMD 2.8, último mês)")
ax.legend(fontsize=9, frameon=False, ncol=2)
salvar(fig, "02_detentores_por_titulo")


fig -> viz/figs/02_detentores_por_titulo.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/02_detentores_por_titulo.png')

In [9]:
# fluxo de juros por detentor x titulo: estoque x custo medio 12m por titulo
cm = pd.read_parquet(p / "dados" / "processado" / "rmd_custo_medio.parquet")
c25 = cm[(cm.tabela == "estoque 12m") & (cm.data.dt.year == 2025)]
custo = c25.groupby("escopo").taxa_pct_aa.mean() / 100
e25 = dt[dt.data.dt.year == 2025].groupby(["titulo", "detentor"]).valor_bi.mean()
fluxo = {t: e25.xs(t) * custo.get(t, float("nan"))
         for t in ["LFT", "LTN", "NTN-B", "NTN-F"]}
tab = pd.DataFrame(fluxo).fillna(0)
tab["TOTAL"] = tab.sum(axis=1)
tab = tab.sort_values("TOTAL", ascending=False)
print("Fluxo de juros estimado 2025, por detentor e título (R$ bi):")
print(tab.round(0).to_string())
print(f"\nvalidação: estimado R$ {tab.TOTAL.sum():,.0f} bi vs "
      f"juros apropriados oficiais de 2025: R$ 880 bi")


Fluxo de juros estimado 2025, por detentor e título (R$ bi):
                            LFT   LTN  NTN-B  NTN-F  TOTAL
detentor                                                  
Instituições Financeiras 176.00 57.00  37.00  15.00 285.00
Previdência               88.00 12.00  97.00   2.00 200.00
Fundos de Investimento   154.00 11.00  32.00   2.00 200.00
Não-residentes            16.00 40.00   6.00  24.00  86.00
Outros                    23.00  3.00  12.00   1.00  39.00
Seguradoras               12.00  2.00  13.00   2.00  30.00
Governo                    1.00 10.00  10.00   5.00  25.00
Tesouro Direto             8.00  2.00  10.00   0.00  21.00

validação: estimado R$ 886 bi vs juros apropriados oficiais de 2025: R$ 880 bi


**O que a fatiagem mostra (e o 2.7 não):**

- a **previdência** concentra as **NTN-Bs** (47% do título de inflação; 34% em
  2011) — e o seu passivo é indexado à inflação: é hedge estrutural, não aposta;
- os **não-residentes** concentram as **pré-fixadas** (26% das LTN, 40% das
  NTN-F) — a aposta estrangeira no juro nominal;
- **bancos e fundos** concentram as **LFTs** (69%) — a rolagem indexada à
  Selic, o título líquido de curto prazo;
- o **Tesouro Direto** (pessoas físicas) é marginal: 2,3% das LFTs, 6,1% das
  NTN-Bs, ~R$ 21 bi de juros/ano (2,4% do total).

Validação: as duas decomposições concordam — o 2.8 agregado por grupo reproduz
o 2.7 (32,9 / 22,8 / 20,8 / 10,3%) e a estimativa de juros por título fecha com
o oficial (R$ 886 bi estimados vs R$ 880 bi).


## Quem, pelo nome: a carteira dos fundos (CVM CDA)

A CVM publica a carteira mensal de cada fundo de investimento, com o nome
(`DENOM_SOCIAL`) e a posição em títulos públicos. É o detalhe que o RMD 2.7
não dá: os **fundos que carregam a dívida, por nome**. Fonte: CDA, bloco BLC_1.


In [10]:
cvm = pd.read_parquet(p / "dados" / "processado" / "cvm_fundos_titulos.parquet")
tot = cvm.valor_bi.sum()
print(f"títulos públicos em fundos (CVM, {cvm.mes.max():%m/%Y}): R$ {tot:,.0f} bi "
      f"em {cvm.fundo.nunique():,} fundos")
top = cvm.groupby("fundo").valor_bi.sum().nlargest(12)
print("\nmaiores fundos por posição em títulos públicos (R$ bi):")
for nome, v in top.items():
    print(f"  {v:8.1f}  {nome[:72]}")


títulos públicos em fundos (CVM, 10/2026): R$ 3,156 bi em 5,993 fundos

maiores fundos por posição em títulos públicos (R$ bi):
     274.4  BRASILPREV TOP TPF FUNDO DE INVESTIMENTO FINANCEIRO RENDA FIXA RESPONSAB
     153.9  BB RF IV FUNDO DE INVESTIMENTO FINANCEIRO RENDA FIXA LONGO PRAZO RESP LI
     105.3  CAIXA PÓS FIXADO PREVINVEST PREVIDENCIÁRIO FUNDO DE INVESTIMENTO FINANCE
      81.7  FUNDO DE INVESTIMENTO FINANCEIRO RENDA FIXA CFIN TÍTULOS PÚBLICOS PLENAS
      50.6  CAIXA GESTÃO PREVINVEST PREVIDENCIÁRIO FUNDO DE INVESTIMENTO FINANCEIRO 
      50.3  FOX SOBERANO II FUNDO DE INVESTIMENTO FINANCEIRO RENDA FIXA - RESPONSABI
      49.2  SPECIAL RENDA FIXA REFERENCIADO DI FUNDO DE INVESTIMENTO FINANCEIRO RESP
      42.1  BB EXTRAMERCADO EXCLUSIVO FAT FUNDO DE INVESTIMENTO FINANCEIRO RENDA FIX
      41.4  BB TOP RENDA FIXA SIMPLES FUNDO DE INVESTIMENTO FINANCEIRO RESPONSABILID
      35.0  ITAÚ VERSO A RENDA FIXA REFERENCIADO DI FUNDO DE INVESTIMENTO FINANCEIRO
      33.5  BEM TPF FI

In [11]:
# por trás dos fundos: os grandes bancos e gestoras
def _grupo(n):
    u = n.upper()
    for g in ["BRASILPREV", "BB ", "CAIXA", "ITAÚ", "ITAU", "BRADESCO",
              "SANTANDER", "BTG", "XP"]:
        if g in u:
            return g.strip()
    return "outros"
cvm["grupo"] = cvm.fundo.map(_grupo)
por_grupo = cvm.groupby("grupo").valor_bi.sum().sort_values(ascending=False)
print("posição em títulos públicos por banco/gestora (R$ bi):")
print((por_grupo).round(0).to_string())
print(f"\nvalidação: fundos CVM R$ {tot:,.0f} bi vs Fundos+Previdência do "
      f"RMD 2.7 = R$ 4.098 bi — a diferença (~R$ 940 bi) é a previdência "
      "FECHADA (fundos de pensão patronais, fora do perímetro da CVM)")


posição em títulos públicos por banco/gestora (R$ bi):
grupo
outros       982.00
BB           680.00
CAIXA        419.00
BRASILPREV   394.00
ITAÚ         281.00
BRADESCO     207.00
SANTANDER    148.00
ITAU          32.00
BTG            7.00
XP             5.00

validação: fundos CVM R$ 3,156 bi vs Fundos+Previdência do RMD 2.7 = R$ 4.098 bi — a diferença (~R$ 940 bi) é a previdência FECHADA (fundos de pensão patronais, fora do perímetro da CVM)


**Os nomes, resumidos:** os maiores detentores de títulos públicos entre os
fundos são **fundos de previdência e de renda fixa dos grandes bancos**:
BRASILPREV TOP TPF (R$ 274 bi), BB RF IV (R$ 154 bi), CAIXA PÓS FIXADO
PREVINVEST (R$ 105 bi), CFIN TÍTULOS PÚBLICOS PLENAS (R$ 82 bi). Por grupo:
**BB (R$ 680 bi), Caixa (R$ 419), BrasilPrev (R$ 394), Itaú (R$ 313),
Bradesco (R$ 207), Santander (R$ 148)** — somados, os cinco grandes bancos
respondem por cerca de 2/3 dos títulos públicos em fundos.

Ou seja: o grupo "Fundos + Previdência" do RMD tem nomes — e são, em grande
medida, **veículos dos próprios bancos** (fundos próprios e de previdência
aberta), além dos fundos independentes. O look-through final (o cotista do
fundo) continua não público — mas o nível institucional já é nomeado.


**Achados verificáveis (P2).**

- O **sistema financeiro** — Instituições Financeiras + Fundos de Investimento —
  detém cerca de metade do estoque e é o maior beneficiário estimado do fluxo
  de juros (só as instituições financeiras: ~R$ 330 bi no ano em 2025).
- **Previdência** (fundos de pensão patronais, incluindo planos de servidores e
  trabalhadores privados) é o 2º grupo: ~22–24%.
- **Pessoas físicas** (Tesouro Direto) têm participação pequena: ~3% — o padrão
  de beneficiário é **institucional**, não pulverizado como o discurso do
  "Tesouro Direto democratizou" sugere.
- **Não-residentes** oscilaram entre ~9% e ~14% da carteira média: parcela
  estruturalmente estrangeira, sensível a crises.

**Limitação:** saber *quais* instituições, dentro de cada grupo, detêm quanto
não é publicável (sigilo; Req. 696/2019 negado). O produto deve explicitar
isso — concentração intra-grupo é lacuna informada, não escondida.